# Fooocus Colab Edition

Compatible con el runtime GPU actual de Colab. Ejecuta la celda de arranque con una GPU activa.

La instalación usa un entorno virtual aislado: conserva Torch/CUDA de Colab y evita contaminar el runtime con las dependencias antiguas de Fooocus.

In [ ]:
# @title Arrancar Fooocus de forma estable
BRANCH = 'colab-2.5.6'  # @param {type:"string"}
TUNEL = 'cloudflare'  # @param ["cloudflare", "gradio"]
CACHEAR_MODELOS_EN_DRIVE = False  # @param {type:"boolean"}
EXTRAS_OPCIONALES = False  # @param {type:"boolean"}
ARGUMENTOS_EXTRA = ''  # @param {type:"string"}

import os, re, shlex, shutil, subprocess, sys, threading, queue, time
REPO = 'https://github.com/deleonramiro085/Fooocus.git'
WORKDIR = '/content/Fooocus'
VENV = '/content/fooocus-venv'
PORT = 7865
DRIVE_CACHE = '/content/drive/MyDrive/Fooocus/models'
SUBDIRS = ['checkpoints','loras','inpaint','controlnet','clip_vision','upscale_models','vae','vae_approx','sam','safety_checker']
CF_DEB = 'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb'

gpu = subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader'], capture_output=True, text=True).stdout.strip()
print('GPU:', gpu or 'NO DETECTADA')
if not gpu: raise RuntimeError('Activa una GPU en Entorno de ejecucion > Cambiar tipo de entorno.')

# Clon limpio o actualización fast-forward. No se sobreescriben cambios locales.
if not os.path.isdir(os.path.join(WORKDIR, '.git')):
    shutil.rmtree(WORKDIR, ignore_errors=True)
    subprocess.run(['git','clone','--depth','1','--branch',BRANCH,REPO,WORKDIR], check=True)
else:
    subprocess.run(['git','-C',WORKDIR,'fetch','origin',BRANCH,'--depth','1'], check=False)
    current = subprocess.run(['git','-C',WORKDIR,'branch','--show-current'], capture_output=True, text=True).stdout.strip()
    if current != BRANCH:
        subprocess.run(['git','-C',WORKDIR,'checkout',BRANCH], check=True)
    status = subprocess.run(['git','-C',WORKDIR,'status','--porcelain'], capture_output=True, text=True).stdout.strip()
    if not status:
        subprocess.run(['git','-C',WORKDIR,'reset','--hard',f'origin/{BRANCH}'], check=True)
    else:
        print('Cambios locales detectados: se conserva la copia y no se hace reset.')
os.chdir(WORKDIR)

# Venv con system-site-packages: hereda el Torch CUDA de Colab, pero aísla el resto.
VENV_PY = os.path.join(VENV, 'bin', 'python')
if not os.path.exists(VENV_PY): subprocess.run([sys.executable,'-m','venv','--system-site-packages',VENV], check=True)
subprocess.run([VENV_PY,'-m','pip','install','--upgrade','pip','setuptools','wheel'], check=True)

if shutil.which('aria2c') is None:
    subprocess.run(['apt-get','update','-qq'], check=False)
    subprocess.run(['apt-get','install','-y','-qq','aria2'], check=False)

if CACHEAR_MODELOS_EN_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    for sub in SUBDIRS:
        target = os.path.join(DRIVE_CACHE, sub); os.makedirs(target, exist_ok=True)
        local = os.path.join(WORKDIR,'models',sub)
        if not os.path.islink(local):
            shutil.rmtree(local, ignore_errors=True); os.symlink(target, local)

tunel_url = None
if TUNEL == 'cloudflare':
    if shutil.which('cloudflared') is None:
        subprocess.run(['wget','-q','-O','/content/cloudflared.deb',CF_DEB], check=False)
        subprocess.run(['dpkg','-i','/content/cloudflared.deb'], check=False, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    if shutil.which('cloudflared'):
        p = subprocess.Popen(['cloudflared','tunnel','--no-autoupdate','--url',f'http://127.0.0.1:{PORT}'], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        lines = queue.Queue()
        threading.Thread(target=lambda: [lines.put(x) for x in p.stdout], daemon=True).start()
        deadline = time.monotonic() + 45
        while time.monotonic() < deadline and tunel_url is None:
            try:
                match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', lines.get(timeout=1))
                if match: tunel_url = match.group(0)
            except queue.Empty: pass
        if tunel_url: print('URL PUBLICA:', tunel_url)
        else:
            print('Cloudflare no respondio en 45 s; se usa --share.')
            p.kill()

cmd = [VENV_PY,'-u','entry_with_update.py','--skip-update','--preset','default','--disable-analytics','--port',str(PORT)]
cmd += ['--listen','127.0.0.1'] if tunel_url else ['--share']
if EXTRAS_OPCIONALES: cmd.append('--install-optional')
cmd += shlex.split(ARGUMENTOS_EXTRA)
print('>', ' '.join(cmd), flush=True)
subprocess.run(cmd, check=False)


In [ ]:
# @title Diagnostico
import importlib.metadata as md, platform, shutil
print('python', platform.python_version())
try:
 import torch; print('torch', torch.__version__, '| cuda', torch.version.cuda); print('gpu', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'SIN GPU')
except Exception as e: print('torch:', e)
print('aria2c', shutil.which('aria2c') or 'no instalado')
print('cloudflared', shutil.which('cloudflared') or 'no instalado')
for p in ('gradio','gradio_client','numpy','transformers','huggingface_hub','pydantic','fastapi','starlette','websockets','pygit2'):
 try: print(p, md.version(p))
 except Exception: print(p, 'no instalado')


## Notas

Si aparece `CUDA out of memory`, deja `ARGUMENTOS_EXTRA = ''`; la configuración normal es más segura para una T4. Si Cloudflare falla, el código cambia automáticamente a `--share`.